# QMV Index & Backtesting

This notebook rigorously evaluates the production model using a rolling-origin backtest to detect concept drift, and then constructs the final 0-100 QMV Index.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor

from src.backtest.rolling import rolling_origin_cv
from src.backtest.index import compute_qmv_index
from src.models.metrics import calculate_metrics


## 1. Rolling-Origin Backtest
We retrain the model on expanding history and predict forward in chunks to simulate production behavior.

In [ ]:
df = pd.read_parquet('data/processed/features_t3.parquet').sort_values('time')
feature_cols = [c for c in df.columns if c not in ['id', 'title', 'url', 'by', 'time', 'type', 'target', 'domain']]
mc = [1 if col in ['v3', 'pts_t3', 'comms_t3', 'author_hist_median'] else 0 for col in feature_cols]

model = HistGradientBoostingRegressor(max_iter=100, learning_rate=0.05, max_depth=5, monotonic_cst=mc, random_state=42)

cv_results = rolling_origin_cv(df, model, feature_cols, initial_train_days=30, step_days=15)
display(cv_results)

plt.figure(figsize=(10, 4))
plt.plot(cv_results['train_end'], cv_results['RMSE'], marker='o', label='RMSE')
plt.title('Backtest Error Over Time (Concept Drift Check)')
plt.ylabel('RMSE (Log Target)')
plt.xlabel('Validation Start Date')
plt.grid(True)
plt.show()

## 2. QMV Index Construction & Monotonicity
We convert the raw log predictions into a 0-100 percentile index using a trailing 30-day window. This makes the score immediately interpretable.

In [ ]:
split = int(len(df)*0.5)
train, test = df.iloc[:split], df.iloc[split:].copy()
X_tr = np.array(train[feature_cols].values, dtype=np.float64)
y_tr = np.array(train['target'].values, dtype=np.float64)
X_te = np.array(test[feature_cols].values, dtype=np.float64)
model.fit(X_tr, y_tr)
test['pred_value'] = model.predict(X_te)

test = compute_qmv_index(test)

test['qmv_decile'] = pd.qcut(test['qmv_index'], 10, labels=False, duplicates='drop')

plt.figure(figsize=(10, 5))
sns.boxplot(x='qmv_decile', y='target', data=test, palette='viridis')
plt.title('Monotonicity: Higher QMV Deciles clearly separate higher realized attention')
plt.xlabel('QMV Decile (0=Lowest, 9=Highest)')
plt.ylabel('Realized Final Log Target')
plt.show()

## 3. Rank Stability (T=1h vs T=6h)
How much does the QMV index change for a story between 1 hour and 6 hours?

In [ ]:
try:
    df1 = pd.read_parquet('data/processed/features_t1.parquet').sort_values('time')
    df6 = pd.read_parquet('data/processed/features_t6.parquet').sort_values('time')
    
    X1_tr = np.array(df1.iloc[:split][feature_cols].values, dtype=np.float64)
    y1_tr = np.array(df1.iloc[:split]['target'].values, dtype=np.float64)
    X1_te = np.array(df1.iloc[split:][feature_cols].values, dtype=np.float64)
    model.fit(X1_tr, y1_tr)
    t1_preds = model.predict(X1_te)
    
    X6_tr = np.array(df6.iloc[:split][feature_cols].values, dtype=np.float64)
    y6_tr = np.array(df6.iloc[:split]['target'].values, dtype=np.float64)
    X6_te = np.array(df6.iloc[split:][feature_cols].values, dtype=np.float64)
    model.fit(X6_tr, y6_tr)
    t6_preds = model.predict(X6_te)
    
    df_rank = pd.DataFrame({'T1': t1_preds, 'T6': t6_preds})
    corr = df_rank.corr(method='spearman').iloc[0, 1]
    
    plt.figure(figsize=(6, 6))
    plt.scatter(df_rank['T1'], df_rank['T6'], alpha=0.2, s=5)
    plt.title(f'Rank Stability: T=1h vs T=6h (Spearman: {corr:.3f})')
    plt.xlabel('Predicted Log Value (T=1h)')
    plt.ylabel('Predicted Log Value (T=6h)')
    plt.show()
except Exception as e:
    print("Could not run rank stability:", e)


## 4. Economic Framing
Simulating a trading strategy: 'Buy' at the predicted value, 'Realize' at the 72h value.

In [ ]:
test['realized_attention'] = np.expm1(test['target'])
test['predicted_attention'] = np.expm1(test['pred_value'])

test['roi_ratio'] = test['realized_attention'] / (test['predicted_attention'] + 1)
roi_by_decile = test.groupby('qmv_decile')['roi_ratio'].median()
display(pd.DataFrame({'Median ROI Ratio': roi_by_decile}))

print("Disclaimer: This 'ROI' represents engagement capture efficiency. It assumes we pay a price equal to predicted attention, which is highly stylized and ignores real-world pricing liquidity.")

## 5. Robustness Checks
What happens to the RMSE if we remove the top author or shift the window?

In [ ]:
top_author = test['by'].value_counts().index[0]
baseline_rmse = calculate_metrics(test['target'], test['pred_value'])['RMSE']

test_no_author = test[test['by'] != top_author]
rmse_no_author = calculate_metrics(test_no_author['target'], test_no_author['pred_value'])['RMSE']

print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"RMSE without Top Author ({top_author}): {rmse_no_author:.4f} (Drop: {rmse_no_author - baseline_rmse:.4f})")